# Training suara waifu (GPT-SoVITS) di Google Colab

**Persiapan (sekali saja):**
1. Upload `char001.zip` (dari `E:\Pembuatan Waifu\data\voice_samples\`) ke Google Drive, ke folder `MyDrive/waifu_voice/`.
2. Di Colab: menu **Runtime → Change runtime type → T4 GPU**.
3. Jalankan sel satu per satu dari atas.

Hasil akhir (model + audio referensi) disimpan ke `MyDrive/waifu_voice/output/`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Mount Google Drive & atur nama

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SPEAKER = 'char001'                                   # harus sama dengan nama zip
DRIVE_DIR = '/content/drive/MyDrive/waifu_voice'
ZIP_PATH = f'{DRIVE_DIR}/{SPEAKER}.zip'
DATA_DIR = '/content/dataset'
import os; assert os.path.exists(ZIP_PATH), f'Tidak ditemukan: {ZIP_PATH}'
print('OK', ZIP_PATH)

## 2. Siapkan dataset (unzip → WAV 32 kHz mono → file .list)

In [ ]:
import os, subprocess, zipfile, glob, shutil, json, wave

shutil.rmtree(DATA_DIR, ignore_errors=True)
os.makedirs(f'{DATA_DIR}/wavs', exist_ok=True)
zipfile.ZipFile(ZIP_PATH).extractall(DATA_DIR)

raw = {os.path.splitext(os.path.basename(p))[0]: p for p in glob.glob(f'{DATA_DIR}/raw/*')}
lines_out, total, durs = [], 0.0, {}
for line in open(f'{DATA_DIR}/{SPEAKER}.list', encoding='utf-8'):
    line = line.strip()
    if not line: continue
    rel, spk, lang, text = line.split('|', 3)
    stem = os.path.splitext(os.path.basename(rel))[0]
    out = f'{DATA_DIR}/wavs/{stem}.wav'
    r = subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', raw[stem],
                        '-ac', '1', '-ar', '32000', '-sample_fmt', 's16', out])
    if r.returncode != 0 or not text.strip():
        print('lewati', stem); continue
    with wave.open(out) as w: d = w.getnframes() / w.getframerate()
    durs[stem] = (d, text); total += d
    lines_out.append(f'{out}|{spk}|{lang.upper()}|{text}')

LIST_PATH = f'{DATA_DIR}/{SPEAKER}_abs.list'
open(LIST_PATH, 'w', encoding='utf-8').write('\n'.join(lines_out) + '\n')
shutil.rmtree(f'{DATA_DIR}/raw')

# pilih audio referensi: klip 4-8 detik dengan teks paling panjang
cand = [(len(t), s) for s, (d, t) in durs.items() if 4 <= d <= 8 and not any(c in t for c in '♡…―ー')] or \
       [(len(t), s) for s, (d, t) in durs.items() if 3 <= d <= 10]
REF_STEM = max(cand)[1]
print(f'{len(lines_out)} klip, {total/60:.1f} menit')
print('LIST :', LIST_PATH)
print('REF  :', REF_STEM, durs[REF_STEM])

Dengarkan audio referensi. Kalau kurang bagus, ganti `REF_STEM` dengan nomor lain (misal `'00042'`) lalu jalankan sel ini lagi.

In [ ]:
from IPython.display import Audio, display
print(durs[REF_STEM][1])
display(Audio(f'{DATA_DIR}/wavs/{REF_STEM}.wav'))

## 3. Install GPT-SoVITS (± 5–10 menit)

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/RVC-Boss/GPT-SoVITS.git
%cd /content/GPT-SoVITS
!apt-get -qq install -y ffmpeg > /dev/null
!pip install -q -r extra-req.txt --no-deps 2>/dev/null || true

In [ ]:
%%bash
# Sel %%bash: `$` tidak diganti variabel Python oleh IPython (beda dengan baris `!`).
# `pip install -r requirements.txt` batal total kalau 1 paket gagal (mis. python_mecab_ko untuk
# bahasa Korea tidak bisa di-build di Python 3.13). Jadi: buang mecab & pyopenjtalk, pasang satu per satu.
# pyopenjtalk-plus = pengganti pyopenjtalk yang punya wheel siap pakai (nama modul sama).
cd /content/GPT-SoVITS
grep -vE '^\s*(#|$)' requirements.txt | grep -viE 'mecab|pyopenjtalk' > /tmp/req.txt
while read -r r; do pip install -q "$r" >/dev/null 2>&1 || echo "GAGAL: $r"; done < /tmp/req.txt
pip install -q opencc pyopenjtalk-plus x_transformers
# gradio 4.x memanggil TemplateResponse gaya lama; starlette terbaru sudah menghapusnya
# -> 'TypeError: unhashable type: dict' saat membuka link gradio. Kunci ke versi yang cocok.
pip install -q "fastapi==0.115.12" "starlette==0.46.2"
# transformers versi baru (5.x) tidak cocok dengan GPT-SoVITS -> 'Could not find BertForMaskedLM'
pip install -q "transformers>=4.43,<=4.50"
python -c "import gradio, fastapi, starlette; print('gradio', gradio.__version__, '| fastapi', fastapi.__version__, '| starlette', starlette.__version__)"
python -c "import pyopenjtalk; print('pyopenjtalk OK:', pyopenjtalk.g2p('こんにちは'))"

In [ ]:
# cek semua modul penting sudah ada sebelum lanjut
import importlib
mods = ['opencc', 'pyopenjtalk', 'gradio', 'torch', 'torchaudio', 'transformers', 'librosa',
        'pytorch_lightning', 'jieba_fast', 'g2p_en', 'cn2an', 'pypinyin', 'ffmpeg']
missing = []
for m in mods:
    try: importlib.import_module(m)
    except Exception as e: missing.append(f'{m}: {type(e).__name__}: {e}')
print('SEMUA OK' if not missing else 'BELUM TERPASANG: ' + ' | '.join(missing))
import torch; print('CUDA:', torch.cuda.is_available(), '(harus True)')
import transformers; from transformers import AutoModelForMaskedLM, BertForMaskedLM
print('transformers', transformers.__version__, '- BertForMaskedLM OK')
# modul yang dipakai skrip training SoVITS & GPT
!cd /content/GPT-SoVITS/GPT_SoVITS && python -c "import sys; sys.path[:0]=['.','..']; import module.models, AR.models.t2s_lightning_module; print('IMPORT TRAINING OK')"

In [ ]:
# model pretrained (dari HuggingFace lj1995/GPT-SoVITS)
from huggingface_hub import snapshot_download
# hanya unduh file untuk versi v2 (bukan seluruh repo yang berisi banyak versi)
snapshot_download('lj1995/GPT-SoVITS', local_dir='/content/GPT-SoVITS/GPT_SoVITS/pretrained_models',
                  allow_patterns=['chinese-hubert-base/*', 'chinese-roberta-wwm-ext-large/*', 'gsv-v2final-pretrained/*'])
!ls /content/GPT-SoVITS/GPT_SoVITS/pretrained_models

In [ ]:
# Susun kamus Jepang sekali di sini. Kalau tidak, 2 proses WebUI menyusunnya bersamaan -> "Bus error"
%cd /content/GPT-SoVITS
!python -c "import sys; sys.path.insert(0, 'GPT_SoVITS'); from text import japanese; print(japanese.g2p('こんにちは、マスター'))"

## 4. Jalankan WebUI & training

Sel di bawah akan memunculkan link **`https://xxxx.gradio.live`**. Buka link itu, lalu di tab **1-GPT-SoVITS-TTS**:

1. **Experiment/model name:** `char001` (persis, tanpa spasi). Untuk **model version**, pilih **`v2`** (hanya model pretrained v2 yang diunduh).
2. Tab **1A - Dataset formatting**:
   - *Text labelling file*: `/content/dataset/char001_abs.list`
   - *Audio dataset folder*: **kosongkan** (path di file .list sudah absolut)
   - *GPU number*: ubah `0-0` menjadi **`0`** (satu proses saja, lebih stabil)
   - Klik **One-click formatting**, lalu tunggu sampai selesai **tanpa error**. Jangan mulai training sebelum langkah ini berhasil.
3. Tab **1B - Fine-tuned training**:
   - **SoVITS training**: batch size `6`, total epochs `8`, lalu klik *Start SoVITS training*. Tunggu selesai.
   - **GPT training**: batch size `6`, total epochs `15`, lalu klik *Start GPT training*. Tunggu selesai.
4. (Opsional) Uji suara di tab **1C - Inference**.

Setelah training selesai, **stop sel ini** (tombol ■), lalu jalankan sel 5.

In [ ]:
%cd /content/GPT-SoVITS
!is_share=True python webui.py en_US

## 5. Simpan model ke Google Drive

In [ ]:
import glob, os, shutil
OUT = f'{DRIVE_DIR}/output'
os.makedirs(OUT, exist_ok=True)
def newest(pattern):
    fs = sorted(glob.glob(pattern, recursive=True), key=os.path.getmtime)
    return fs[-1] if fs else None
gpt = newest(f'/content/GPT-SoVITS/GPT_weights*/{SPEAKER}*.ckpt')
sov = newest(f'/content/GPT-SoVITS/SoVITS_weights*/{SPEAKER}*.pth')
assert gpt and sov, 'Model belum ada — pastikan training SoVITS & GPT sudah selesai'
for f in (gpt, sov): shutil.copy(f, OUT)
shutil.copy(f'{DATA_DIR}/wavs/{REF_STEM}.wav', f'{OUT}/ref.wav')
open(f'{OUT}/ref.txt', 'w', encoding='utf-8').write(durs[REF_STEM][1])
info = {'speaker': SPEAKER, 'gpt': os.path.basename(gpt), 'sovits': os.path.basename(sov),
        'ref_audio': 'ref.wav', 'ref_text': durs[REF_STEM][1], 'ref_lang': 'ja',
        'version_dir': os.path.basename(os.path.dirname(sov))}
json.dump(info, open(f'{OUT}/model_info.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
print(json.dumps(info, ensure_ascii=False, indent=2))
!ls -la "{OUT}"

Selesai! Unduh folder `MyDrive/waifu_voice/output/` ke laptop, misalnya ke `E:\Pembuatan Waifu\voice_models\char001\`, lalu beri tahu Claude untuk menyambungkannya ke `speech/voicefox.py`.